In [1]:
from langchain_groq import ChatGroq

In [2]:
pip install -qU langchain_community
pip install -qU langchai

SyntaxError: invalid syntax (2360289572.py, line 1)

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(
    temperature = 0,
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name='openai/gpt-oss-120b'
)
response=llm.invoke("The first person to land on moon was ...")
print(response.content)

The first person to set foot on the Moon was **Neil Armstrong**, who did so on July 20 1945 (Apollo 11 mission).


In [4]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://candidat.francetravail.fr/offres/recherche?lieux=69266&offresPartenaires=true&rayon=10&secteursActivite=64&tri=0")
page_data = loader.load().pop().page_content
print(page_data)

/var/folders/c0/mp40hw7557lfvpt1ysnpj2r80000gn/T/ipykernel_33353/751917212.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Offres d'emploi Villeurbanne (69100) | France Travail

ImprimerBesoin d'aide sur la recherche d'offres d'emploi ?Métier, compétence, mot-clé, numéro d'offreLieu de travailVilleurbanne (69100)Distance autour du lieu choisiLieu exact5 km10 km20 km30 km40 km50 km60 km70 km80 km90 km100 km
Rechercher des offres
Afficher uniquement

les offres France Travail
 Cette action recharge la liste des offresQui sont nos partenaires ?Afficher les filtres de Date de création de l'offreDate de création de l'offreUn jourTrois joursUne semaineDeux semainesUn moisToutes les offres
Appliquer le filtre de date de création de l'offreAfficher les filtres de types de Contrat
Type de contratCDI (458)CDD (29)IntérimCDI IntérimaireSaisonnierContrat apprentissage (3)Cont. professionnalisation (2)Prépa.opérationnel.emploiCDI de chantier ou d'opérationContrat d'Engagement EducatifContrat intermittentContrat pacteContrat d'usageCUI - CAECUI - CIEEngagement à servir dans la réserveFranchiseInsertion par l'activ.éco.P

In [5]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
    """
    ### SCRAPED TEXT FROM WEBSITE:
    {page_data}
    ### INSTRUCTION: 
    The scraped text is from the career's page of a website.
    Your job is to extract the job postings and return them in JSON formar containing this following keys: 
    'role', 'experience', 'skills', and 'description' . 
    Only return the valid JSON
    ### VALID JSON (NO PREAMBLE):
    """    
)

chain_extract =  prompt_extract | llm
res = chain_extract.invoke(input={'page_data':page_data })
print(res.content)

[
  {
    "role": "Commis de cuisine",
    "experience": null,
    "skills": [],
    "description": ""
  },
  {
    "role": "Directeur d'Agence",
    "experience": null,
    "skills": [],
    "description": ""
  },
  {
    "role": "Commis de cuisine",
    "experience": null,
    "skills": [],
    "description": ""
  },
  {
    "role": "Responsable d'agence de travail temporaire",
    "experience": null,
    "skills": [],
    "description": ""
  },
  {
    "role": "Délégué commercial / Déléguée commerciale en services",
    "experience": null,
    "skills": [],
    "description": ""
  },
  {
    "role": "Gestionnaire de Paie & ADP",
    "experience": null,
    "skills": [],
    "description": ""
  },
  {
    "role": "Technicien datacenter débutant",
    "experience": null,
    "skills": [],
    "description": ""
  },
  {
    "role": "Gestionnaire back office crédits",
    "experience": null,
    "skills": [],
    "description": ""
  },
  {
    "role": "Gestionnaire Recouvrement",
    "e

In [6]:
type(res.content)

str

In [7]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

[{'role': 'Commis de cuisine',
  'experience': None,
  'skills': [],
  'description': ''},
 {'role': "Directeur d'Agence",
  'experience': None,
  'skills': [],
  'description': ''},
 {'role': 'Commis de cuisine',
  'experience': None,
  'skills': [],
  'description': ''},
 {'role': "Responsable d'agence de travail temporaire",
  'experience': None,
  'skills': [],
  'description': ''},
 {'role': 'Délégué commercial / Déléguée commerciale en services',
  'experience': None,
  'skills': [],
  'description': ''},
 {'role': 'Gestionnaire de Paie & ADP',
  'experience': None,
  'skills': [],
  'description': ''},
 {'role': 'Technicien datacenter débutant',
  'experience': None,
  'skills': [],
  'description': ''},
 {'role': 'Gestionnaire back office crédits',
  'experience': None,
  'skills': [],
  'description': ''},
 {'role': 'Gestionnaire Recouvrement',
  'experience': None,
  'skills': [],
  'description': ''},
 {'role': 'Chef de projet / Cheffe de projet (Project Management Officer)'

In [8]:
type(json_res)

list

In [9]:
import pandas as pd

df = pd.read_csv('app/resource/my_portfolio.csv')
df.head()

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio


In [10]:
import chromadb
import uuid

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents = row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

In [11]:
links =  collection.query(query_texts=["Experience in Python", "Experience in React"], n_results=2).get('metadatas', [])
links 

[[{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/python-portfolio'}],
 [{'links': 'https://example.com/react-portfolio'},
  {'links': 'https://example.com/react-native-portfolio'}]]

In [12]:
job = json_res
job['skills']

TypeError: list indices must be integers or slices, not str

In [ ]:
prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}
        
        ### INSTRUCTION:
        You are Idrissa, a business development executive at an anonymous engineering company. The company is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools.
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability,
        process optimization, cost reduction, and heightened overall efficiency.
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of the company
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase the company's portfolio: {link_list}
        Remember you are Idrissa, business development executive at an anonymous engineering company.
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):
        
        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)